In [1]:
import ase
import numpy as np
import pyscf
import time
import os
from pyscf.scf import hf
from pyscf import gto, df, lib
from pyscf.gto import mole
import scipy

#!/usr/bin/env python3
import torch
from equiv_dens.training.parse_command_line_arguments import parse_command_line_arguments
from equiv_dens.training.errors import ErrorDict
from equiv_dens.data.density_dataset import AtomsDensityData
from equiv_dens.data.hamiltonian_dataset import seeded_random_split
from equiv_dens.utils.grids import cubical_grid, cubical_sampling,\
    dftpy_grid, CubicalGrid, spherical_grid, spherical_radial_sampling
from equiv_dens.training.model_loader import load_model
import equiv_dens.utils.base as utils
from equiv_dens.utils import orbitals
from functools import partial

%load_ext autoreload
%autoreload 2

/home/mihail/anaconda3/envs/equiv_dens/lib/python3.7/site-packages/pyscf/lib/misc.py:47: H5pyDeprecationWarning: Using default_file_mode other than 'r' is deprecated. Pass the mode to h5py.File() instead.
  h5py.get_config().default_file_mode = 'a'


Use "numpy" for Fourier Transform


In [82]:
dat_types = ['valid', 'test']
for dat_type in dat_types:
    data = np.load('datasets/ethanol_pyscf_def2svp_dft_f_' + dat_type + '.npy', allow_pickle=True)
    aux_sets = ['augccpvqzjkfit', 'augccpvdzjkfit', 'ccpvdzjkfit']
    for auxbasis in aux_sets:
        for i in range(len(data)):
            print(i)
            calc = data[i][1]
            mol = mole.unpack(data[i][0])
            mol.build()
            print(calc.keys())
            mo_coeff = calc['mo_coeff']
            mo_occ = calc['mo_occ']
            dm1 = hf.make_rdm1(mo_coeff, mo_occ)

            # Define the auxiliary fitting basis for 3-center integrals. Use the function
            # make_auxmol to construct the auxiliary Mole object (auxmol) which will be
            # used to generate integrals.
            auxmol = df.addons.make_auxmol(mol, auxbasis)

            # ints_3c is the 3-center integral tensor (ij|P), where i and j are the
            # indices of AO basis and P is the auxiliary basis
            ints_3c2e = df.incore.aux_e2(mol, auxmol, intor='int3c2e')
            ints_2c2e = auxmol.intor('int2c2e')

            nao = mol.nao
            naux = auxmol.nao

            # Compute the DF coefficients (df_coef) and the DF 2-electron (df_eri)
            df_coef = scipy.linalg.solve(ints_2c2e, ints_3c2e.reshape(nao*nao, naux).T)
            df_coef = df_coef.reshape(naux, nao, nao)
            df_basis = lib.einsum('Pij,ij->P', df_coef, dm1)

            mol.basis = auxbasis 
            mol.build()
            calc['df_coeff'] = df_basis
            calc['auxbasis'] = auxbasis
        np.save('datasets/ethanol_pyscf_def2svp_dft_f_df_' + auxbasis + '_' + dat_type + '.npy', data, allow_pickle=True)

0
dict_keys(['mo_coeff', 'mo_occ', 'energy', 'forces'])
1
dict_keys(['mo_coeff', 'mo_occ', 'energy', 'forces'])
2
dict_keys(['mo_coeff', 'mo_occ', 'energy', 'forces'])
3
dict_keys(['mo_coeff', 'mo_occ', 'energy', 'forces'])
4
dict_keys(['mo_coeff', 'mo_occ', 'energy', 'forces'])
5
dict_keys(['mo_coeff', 'mo_occ', 'energy', 'forces'])
6
dict_keys(['mo_coeff', 'mo_occ', 'energy', 'forces'])
7
dict_keys(['mo_coeff', 'mo_occ', 'energy', 'forces'])
8
dict_keys(['mo_coeff', 'mo_occ', 'energy', 'forces'])
9
dict_keys(['mo_coeff', 'mo_occ', 'energy', 'forces'])
10
dict_keys(['mo_coeff', 'mo_occ', 'energy', 'forces'])
11
dict_keys(['mo_coeff', 'mo_occ', 'energy', 'forces'])
12
dict_keys(['mo_coeff', 'mo_occ', 'energy', 'forces'])
13
dict_keys(['mo_coeff', 'mo_occ', 'energy', 'forces'])
14
dict_keys(['mo_coeff', 'mo_occ', 'energy', 'forces'])
15
dict_keys(['mo_coeff', 'mo_occ', 'energy', 'forces'])
16
dict_keys(['mo_coeff', 'mo_occ', 'energy', 'forces'])
17
dict_keys(['mo_coeff', 'mo_occ', 'ener

In [2]:
args, hyperparam_args = parse_command_line_arguments(arg_file='ethanol_dens_001_mae_test.txt')

print('type dtype', type(args.dtype))
args.fix_arguments = True
print('args np dir', args.np_dataset)
# no restart directory specified
directory = args.restart  # load directory name
# load latest checkpoint
checkpoint_path = os.path.join(directory, 'checkpoints')  # checkpoint directory
checkpoint = torch.load(os.path.join(
    checkpoint_path, 'latest_checkpoint.pth'), map_location='cpu')
latest_checkpoint = checkpoint['step']
model_code = checkpoint['ID']  # load ID
step = checkpoint['step']
for arg in vars(checkpoint['args']):
    if args.fix_arguments:
        if arg in hyperparam_args:
            print('loading hyperparam arg', arg)
            setattr(args, arg, getattr(checkpoint['args'], arg))
    else:
        print('loading all arg', arg)
        setattr(args, arg, getattr(checkpoint['args'], arg))
restore = True

args.best_model_path = 'best_' + model_code + '.pth'
print('best_model_path', args.best_model_path)

print('model code:', model_code)
# determine whether GPU is used for training
print('args use gpu', args.use_gpu)
args.use_gpu = False
# load dataset(s)
print("loading density from" + str(args.dens_dataset) + "...")
print("loading atoms from" + args.np_dataset + "...")

args.verbose = 0
args.use_gpu = False
args.radii_adjust = True 
if args.cube_grid:
    grid_origin = args.cube_origin
    grid_extent = np.array([args.cube_extent] * 3)
    grid_fn = partial(cubical_grid, nx=args.cube_size, ny=args.cube_size, nz=args.cube_size,
                      extent=grid_extent,
                      origin=np.array([grid_origin] * 3))
    sampling_fn = cubical_sampling
else:
    grid_fn = partial(spherical_grid, level=2)
    sampling_fn = partial(spherical_radial_sampling, rotate=False)
    grid_origin = 0
    grid_extent = None
    
dataset = AtomsDensityData(np_path=args.np_dataset, density_path=args.dens_dataset,
                           orbitals_path=args.orbitals_file,
                           density_n_samp=10000000000,
                           required_properties=['density'],
                           center_positions=False,
                           radial_coeffs_file=args.radial_coeffs_file,
                           dtype=args.dtype,
                           grid_fn=grid_fn,
                           sampling_fn=sampling_fn,
                           grid_extent=grid_extent,
                           grid_origin=grid_origin,
                           verbose=args.verbose,
                           radii_adjust=args.radii_adjust)
print(args.dtype)

type dtype <class 'torch.dtype'>
args np dir datasets/ethanol_dft_train.npy
loading hyperparam arg activation
loading hyperparam arg order
loading hyperparam arg mixing_order
loading hyperparam arg order_en
loading hyperparam arg mixing_order_en
loading hyperparam arg num_features
loading hyperparam arg num_basis_functions
loading hyperparam arg num_radial_components
loading hyperparam arg num_energy_features
loading hyperparam arg num_modules
loading hyperparam arg num_residual_pre_x
loading hyperparam arg num_residual_post_x
loading hyperparam arg num_residual_pre_vi
loading hyperparam arg num_residual_pre_vj
loading hyperparam arg num_residual_post_v
loading hyperparam arg num_residual_output
loading hyperparam arg num_energy_output
loading hyperparam arg basis_functions
loading hyperparam arg cutoff
loading hyperparam arg orthonormal_basis
loading hyperparam arg expansion_constraint
loading hyperparam arg integral_constraint
loading hyperparam arg integral_scale
loading hyperparam 

In [3]:
split_dens_dataset = args.dens_dataset.split('.')
df_dataset = '.'.join([split_dens_dataset[0] + '_df_augccpvqz', split_dens_dataset[1]]) 

dataset_df = AtomsDensityData(np_path=args.np_dataset, density_path=df_dataset,
                           orbitals_path=args.orbitals_file,
                           density_n_samp=10000000000,
                           required_properties=['density'],
                           center_positions=False,
                           radial_coeffs_file=args.radial_coeffs_file,
                           dtype=args.dtype,
                           grid_fn=grid_fn,
                           sampling_fn=sampling_fn,
                           grid_extent=grid_extent,
                           grid_origin=grid_origin,
                           verbose=args.verbose,
                           radii_adjust=args.radii_adjust,
                           projected_density=True)

Starting atomsdata density init
Some variables
atoms keys dict_keys(['positions', 'energy', 'forces', 'atom_numbers', 'atom_types'])
grid fn functools.partial(<function spherical_grid at 0x7fefe1094400>, level=2)
len atom types 9
atom numbers 6
level 2
grid spec sizes [torch.Size([11208]), torch.Size([11388]), torch.Size([5468])]
finished init


In [34]:
# augccpvdzfit
df_errors = []
for i in range(len(dataset)):
    mol = dataset.get_properties([i])
    mol_df = dataset_df.get_properties([i])
    
    df_error = torch.mean(torch.abs(mol['density'] - mol_df['density']) * mol['coord_weights']) / torch.mean(mol['density'] * mol['coord_weights'])
    print('df_error', i, ':', df_error)
    df_errors.append(df_error)
    
print('average_density error', np.mean(df_errors))

df_error 0 : tensor(0.0128)
df_error 1 : tensor(0.0127)
df_error 2 : tensor(0.0128)
df_error 3 : tensor(0.0130)
df_error 4 : tensor(0.0127)
df_error 5 : tensor(0.0130)
df_error 6 : tensor(0.0129)
df_error 7 : tensor(0.0127)
df_error 8 : tensor(0.0127)
df_error 9 : tensor(0.0127)
df_error 10 : tensor(0.0124)
df_error 11 : tensor(0.0128)
df_error 12 : tensor(0.0131)
df_error 13 : tensor(0.0126)
df_error 14 : tensor(0.0126)
df_error 15 : tensor(0.0126)
df_error 16 : tensor(0.0126)
df_error 17 : tensor(0.0131)
df_error 18 : tensor(0.0125)
df_error 19 : tensor(0.0125)
df_error 20 : tensor(0.0128)
df_error 21 : tensor(0.0130)
df_error 22 : tensor(0.0126)
df_error 23 : tensor(0.0129)
df_error 24 : tensor(0.0130)
df_error 25 : tensor(0.0129)
df_error 26 : tensor(0.0131)
df_error 27 : tensor(0.0131)
df_error 28 : tensor(0.0131)
df_error 29 : tensor(0.0128)
df_error 30 : tensor(0.0126)
df_error 31 : tensor(0.0127)
df_error 32 : tensor(0.0127)
df_error 33 : tensor(0.0127)
df_error 34 : tensor(0.0

In [38]:
# augccpvqzfit
df_errors = []
for i in range(len(dataset)):
    mol = dataset.get_properties([i])
    mol_df = dataset_df.get_properties([i])
    
    df_error = torch.mean(torch.abs(mol['density'] - mol_df['density']) * mol['coord_weights']) / torch.mean(mol['density'] * mol['coord_weights'])
    print('df_error', i, ':', df_error)
    df_errors.append(df_error)
    
print('average_density error', np.mean(df_errors))

df_error 0 : tensor(0.0069)
df_error 1 : tensor(0.0070)
df_error 2 : tensor(0.0069)
df_error 3 : tensor(0.0071)
df_error 4 : tensor(0.0070)
df_error 5 : tensor(0.0070)
df_error 6 : tensor(0.0069)
df_error 7 : tensor(0.0069)
df_error 8 : tensor(0.0071)
df_error 9 : tensor(0.0070)
df_error 10 : tensor(0.0069)
df_error 11 : tensor(0.0071)
df_error 12 : tensor(0.0070)
df_error 13 : tensor(0.0070)
df_error 14 : tensor(0.0070)
df_error 15 : tensor(0.0069)
df_error 16 : tensor(0.0071)
df_error 17 : tensor(0.0071)
df_error 18 : tensor(0.0070)
df_error 19 : tensor(0.0070)
df_error 20 : tensor(0.0071)
df_error 21 : tensor(0.0070)
df_error 22 : tensor(0.0070)
df_error 23 : tensor(0.0070)
df_error 24 : tensor(0.0070)
df_error 25 : tensor(0.0070)
df_error 26 : tensor(0.0071)
df_error 27 : tensor(0.0070)
df_error 28 : tensor(0.0070)
df_error 29 : tensor(0.0070)
df_error 30 : tensor(0.0070)
df_error 31 : tensor(0.0071)
df_error 32 : tensor(0.0069)
df_error 33 : tensor(0.0070)
df_error 34 : tensor(0.0

In [41]:
# def2svpjkfit
df_errors = []
for i in range(len(dataset)):
    mol = dataset.get_properties([i])
    mol_df = dataset_df.get_properties([i])
    
    df_error = torch.mean(torch.abs(mol['density'] - mol_df['density']) * mol['coord_weights']) / torch.mean(mol['density'] * mol['coord_weights'])
    print('df_error', i, ':', df_error)
    df_errors.append(df_error)
    
print('average_density error', np.mean(df_errors))

df_error 0 : tensor(0.0101)
df_error 1 : tensor(0.0101)
df_error 2 : tensor(0.0103)
df_error 3 : tensor(0.0103)
df_error 4 : tensor(0.0100)
df_error 5 : tensor(0.0104)
df_error 6 : tensor(0.0101)
df_error 7 : tensor(0.0100)
df_error 8 : tensor(0.0100)
df_error 9 : tensor(0.0100)
df_error 10 : tensor(0.0102)
df_error 11 : tensor(0.0105)
df_error 12 : tensor(0.0104)
df_error 13 : tensor(0.0103)
df_error 14 : tensor(0.0103)
df_error 15 : tensor(0.0102)
df_error 16 : tensor(0.0101)
df_error 17 : tensor(0.0102)
df_error 18 : tensor(0.0100)
df_error 19 : tensor(0.0103)
df_error 20 : tensor(0.0103)
df_error 21 : tensor(0.0102)
df_error 22 : tensor(0.0100)
df_error 23 : tensor(0.0104)
df_error 24 : tensor(0.0102)
df_error 25 : tensor(0.0101)
df_error 26 : tensor(0.0105)
df_error 27 : tensor(0.0104)
df_error 28 : tensor(0.0104)
df_error 29 : tensor(0.0102)
df_error 30 : tensor(0.0105)
df_error 31 : tensor(0.0101)
df_error 32 : tensor(0.0103)
df_error 33 : tensor(0.0102)
df_error 34 : tensor(0.0

In [44]:

# ccpvdzjkfit
df_errors = []
for i in range(len(dataset)):
    mol = dataset.get_properties([i])
    mol_df = dataset_df.get_properties([i])
    
    df_error = torch.mean(torch.abs(mol['density'] - mol_df['density']) * mol['coord_weights']) / torch.mean(mol['density'] * mol['coord_weights'])
    print('df_error', i, ':', df_error)
    df_errors.append(df_error)
    
print('average_density error', np.mean(df_errors))

df_error 0 : tensor(0.0137)
df_error 1 : tensor(0.0132)
df_error 2 : tensor(0.0132)
df_error 3 : tensor(0.0138)
df_error 4 : tensor(0.0134)
df_error 5 : tensor(0.0139)
df_error 6 : tensor(0.0136)
df_error 7 : tensor(0.0135)
df_error 8 : tensor(0.0131)
df_error 9 : tensor(0.0132)
df_error 10 : tensor(0.0132)
df_error 11 : tensor(0.0134)
df_error 12 : tensor(0.0141)
df_error 13 : tensor(0.0132)
df_error 14 : tensor(0.0133)
df_error 15 : tensor(0.0129)
df_error 16 : tensor(0.0131)
df_error 17 : tensor(0.0138)
df_error 18 : tensor(0.0131)
df_error 19 : tensor(0.0131)
df_error 20 : tensor(0.0133)
df_error 21 : tensor(0.0138)
df_error 22 : tensor(0.0130)
df_error 23 : tensor(0.0135)
df_error 24 : tensor(0.0133)
df_error 25 : tensor(0.0136)
df_error 26 : tensor(0.0136)
df_error 27 : tensor(0.0139)
df_error 28 : tensor(0.0138)
df_error 29 : tensor(0.0134)
df_error 30 : tensor(0.0133)
df_error 31 : tensor(0.0134)
df_error 32 : tensor(0.0136)
df_error 33 : tensor(0.0133)
df_error 34 : tensor(0.0

In [14]:
# augccpvqzfit
df_errors = []
for i in range(len(dataset)):
    mol = dataset.get_properties([i])
    mol_df = dataset_df.get_properties([i])
    
    df_error = torch.mean(torch.abs(mol['density'] - mol_df['density']) * mol['coord_weights']) / torch.mean(mol['density'] * mol['coord_weights'])
    print('df_error', i, ':', df_error)
    df_errors.append(df_error)
    
print('average_density error', np.mean(df_errors))

df_error 0 : tensor(0.0061)
df_error 1 : tensor(0.0062)
df_error 2 : tensor(0.0061)
df_error 3 : tensor(0.0063)
df_error 4 : tensor(0.0062)
df_error 5 : tensor(0.0062)
df_error 6 : tensor(0.0061)
df_error 7 : tensor(0.0061)
df_error 8 : tensor(0.0062)
df_error 9 : tensor(0.0062)
df_error 10 : tensor(0.0061)
df_error 11 : tensor(0.0062)
df_error 12 : tensor(0.0062)
df_error 13 : tensor(0.0062)
df_error 14 : tensor(0.0062)
df_error 15 : tensor(0.0061)
df_error 16 : tensor(0.0063)
df_error 17 : tensor(0.0062)
df_error 18 : tensor(0.0062)
df_error 19 : tensor(0.0062)
df_error 20 : tensor(0.0063)
df_error 21 : tensor(0.0062)
df_error 22 : tensor(0.0062)
df_error 23 : tensor(0.0062)
df_error 24 : tensor(0.0062)
df_error 25 : tensor(0.0062)
df_error 26 : tensor(0.0063)
df_error 27 : tensor(0.0062)
df_error 28 : tensor(0.0062)
df_error 29 : tensor(0.0062)
df_error 30 : tensor(0.0062)
df_error 31 : tensor(0.0062)
df_error 32 : tensor(0.0061)
df_error 33 : tensor(0.0062)
df_error 34 : tensor(0.0

In [29]:
model = load_model(args, dataset)
sample = dataset.get_properties([5])
sample_df = dataset_df.get_properties([5])
results = model(sample)

cg_matrix shape torch.Size([121, 121, 121])
args energy_unit_in kcal
args energy_unit_out kcal
conversions in <function kcal_to_kcal at 0x7fbb74ba0730>
conversions out <function kcal_to_kcal at 0x7fbb74ba0730>
self order [1, 3, 5]
self order [1, 3, 5]
self mixing_order [1, 3, 5]
self mixing_order [1, 3, 5]
creating embedding
init_coeffs None
orbital basis {6: [(6, 1, 0), (6, 1, 0), (6, 1, 0), (6, 1, 0), (6, 1, 0), (6, 1, 0), (6, 1, 0), (6, 1, 0), (6, 1, 0), (6, 1, 0), (6, 1, 0), (6, 1, 1), (6, 1, 1), (6, 1, 1), (6, 1, 1), (6, 1, 1), (6, 1, 1), (6, 1, 1), (6, 1, 1), (6, 1, 2), (6, 1, 2), (6, 1, 2), (6, 1, 2), (6, 1, 2), (6, 1, 2), (6, 1, 3), (6, 1, 3), (6, 1, 3), (6, 1, 3), (6, 1, 4), (6, 1, 4), (6, 1, 4), (6, 1, 5), (6, 1, 5)], 8: [(8, 1, 0), (8, 1, 0), (8, 1, 0), (8, 1, 0), (8, 1, 0), (8, 1, 0), (8, 1, 0), (8, 1, 0), (8, 1, 0), (8, 1, 0), (8, 1, 0), (8, 1, 1), (8, 1, 1), (8, 1, 1), (8, 1, 1), (8, 1, 1), (8, 1, 1), (8, 1, 1), (8, 1, 1), (8, 1, 2), (8, 1, 2), (8, 1, 2), (8, 1, 2), (8, 1

In [31]:
print('ml-density error', torch.mean(torch.abs(sample['density'] - results['density']) * sample['coord_weights']) / torch.mean(sample['density'] * sample['coord_weights']))
print('df-density error', torch.mean(torch.abs(sample['density'] - sample_df['density']) * sample['coord_weights']) / torch.mean(sample['density'] * sample['coord_weights']))
print('ml-df error', torch.mean(torch.abs(sample_df['density'] - results['density']) * sample['coord_weights']) / torch.mean(sample_df['density'] * sample['coord_weights']))

ml-density error tensor(0.0059, grad_fn=<DivBackward0>)
df-density error tensor(0.0070)
ml-df error tensor(0.0102, grad_fn=<DivBackward0>)


In [65]:
print(model.density_repr_model[0].orbital_basis)
vector_coeffs = orbitals.coeffs_dict_to_vector(results, model.density_repr_model[0].orbital_basis, results['atom_numbers'])
print('sph_vector', vector_coeffs['spherical_coeffs'].shape)

{6: [(6, 1, 0), (6, 1, 0), (6, 1, 0), (6, 1, 0), (6, 1, 0), (6, 1, 0), (6, 1, 0), (6, 1, 0), (6, 1, 0), (6, 1, 0), (6, 1, 0), (6, 1, 1), (6, 1, 1), (6, 1, 1), (6, 1, 1), (6, 1, 1), (6, 1, 1), (6, 1, 1), (6, 1, 1), (6, 1, 2), (6, 1, 2), (6, 1, 2), (6, 1, 2), (6, 1, 2), (6, 1, 2), (6, 1, 3), (6, 1, 3), (6, 1, 3), (6, 1, 3), (6, 1, 4), (6, 1, 4), (6, 1, 4), (6, 1, 5), (6, 1, 5)], 8: [(8, 1, 0), (8, 1, 0), (8, 1, 0), (8, 1, 0), (8, 1, 0), (8, 1, 0), (8, 1, 0), (8, 1, 0), (8, 1, 0), (8, 1, 0), (8, 1, 0), (8, 1, 1), (8, 1, 1), (8, 1, 1), (8, 1, 1), (8, 1, 1), (8, 1, 1), (8, 1, 1), (8, 1, 1), (8, 1, 2), (8, 1, 2), (8, 1, 2), (8, 1, 2), (8, 1, 2), (8, 1, 2), (8, 1, 3), (8, 1, 3), (8, 1, 3), (8, 1, 3), (8, 1, 4), (8, 1, 4), (8, 1, 4), (8, 1, 5), (8, 1, 5)], 1: [(1, 1, 0), (1, 1, 0), (1, 1, 0), (1, 1, 0), (1, 1, 0), (1, 1, 1), (1, 1, 1), (1, 1, 1), (1, 1, 1), (1, 1, 2), (1, 1, 2), (1, 1, 2), (1, 1, 2), (1, 1, 3), (1, 1, 3), (1, 1, 3), (1, 1, 4), (1, 1, 4)]}
sph_vector torch.Size([1, 882])


In [63]:
print(dataset_df.density_fitting[5]['df_coeff'].shape)

(882,)


In [78]:
dict_coeffs = orbitals.vector_to_coeffs_dict(vector_coeffs, model.density_repr_model[0].orbital_basis, results['atom_numbers'])
for i in range(len(dict_coeffs['spherical_coeffs'])):
    for key in dict_coeffs['spherical_coeffs'][i].keys():
        print(key)
        print(torch.all(dict_coeffs['spherical_coeffs'][i][key] == results['spherical_coeffs'][i][key]))
        print(torch.all(dict_coeffs['radial_width'][i][key] == results['radial_width'][i][key]))
        print(torch.all(dict_coeffs['radial_scale'][i][key] == results['radial_scale'][i][key]))

(6, 0)
tensor(True)
tensor(True)
tensor(True)
(6, 1)
tensor(True)
tensor(True)
tensor(True)
(6, 2)
tensor(True)
tensor(True)
tensor(True)
(6, 3)
tensor(True)
tensor(True)
tensor(True)
(6, 4)
tensor(True)
tensor(True)
tensor(True)
(6, 5)
tensor(True)
tensor(True)
tensor(True)
(6, 0)
tensor(True)
tensor(True)
tensor(True)
(6, 1)
tensor(True)
tensor(True)
tensor(True)
(6, 2)
tensor(True)
tensor(True)
tensor(True)
(6, 3)
tensor(True)
tensor(True)
tensor(True)
(6, 4)
tensor(True)
tensor(True)
tensor(True)
(6, 5)
tensor(True)
tensor(True)
tensor(True)
(8, 0)
tensor(True)
tensor(True)
tensor(True)
(8, 1)
tensor(True)
tensor(True)
tensor(True)
(8, 2)
tensor(True)
tensor(True)
tensor(True)
(8, 3)
tensor(True)
tensor(True)
tensor(True)
(8, 4)
tensor(True)
tensor(True)
tensor(True)
(8, 5)
tensor(True)
tensor(True)
tensor(True)
(1, 0)
tensor(True)
tensor(True)
tensor(True)
(1, 1)
tensor(True)
tensor(True)
tensor(True)
(1, 2)
tensor(True)
tensor(True)
tensor(True)
(1, 3)
tensor(True)
tensor(True)
t

In [77]:
print('results', results['radial_width'][0][(6,0)])
print('coeffs', dict_coeffs['radial_width'][0][(6,0)])
print('results', results['radial_scale'][0][(6,0)])
print('coeffs', dict_coeffs['radial_scale'][0][(6,0)])

results tensor([[[[-0.2632, -0.3916, -0.5734,  0.9367,  0.9994, -0.8759,  0.9951,
            0.7586, -0.8817, -0.5924, -0.6177]]]], grad_fn=<CopySlices>)
coeffs tensor([[[[-0.2632, -0.5311, -0.5968, -1.7972, -1.7985, -0.3399, -0.4154,
            0.3605, -1.1211, -1.1324, -0.9836]]]], grad_fn=<CatBackward0>)
results tensor([[[[-1.2541, -0.5311, -0.5968, -1.7972, -1.7985, -0.3399, -0.4154,
            0.3605, -1.1211, -1.1324, -0.9836]]]], grad_fn=<CopySlices>)
coeffs tensor([[[[-1.2541, -0.3916, -0.5734,  0.9367,  0.9994, -0.8759,  0.9951,
            0.7586, -0.8817, -0.5924, -0.6177]]]], grad_fn=<CatBackward0>)


In [69]:
print('dict coeffs', dict_coeffs['spherical_coeffs'])
print('results coeffs', results['spherical_coeffs'])

dict coeffs [{(6, 0): tensor([[[[-0.2327,  0.7453,  0.7161, -0.7758, -0.6771,  0.9481,  0.6386,
            1.6447, -0.0046, -0.0721,  0.1405]]]], grad_fn=<CatBackward0>), (6, 1): tensor([[[[-0.0262, -0.2412,  0.3023,  0.1429,  0.1938, -0.0316,  0.0510,
           -0.0828],
          [ 0.0141,  0.1192, -0.2930, -0.0158,  0.2055,  0.0008, -0.0517,
            0.0207],
          [-0.0035, -0.0460, -0.2185,  0.1614,  0.3393, -0.0273, -0.0279,
           -0.0154]]]], grad_fn=<CatBackward0>), (6, 2): tensor([[[[ 0.0823, -0.0361, -0.1014,  0.0336,  0.1042, -0.0125],
          [-0.0771,  0.0240,  0.1286,  0.0100,  0.0828,  0.0328],
          [-0.0668,  0.0156,  0.1191, -0.0378, -0.0253,  0.0145],
          [ 0.0469, -0.0165, -0.0646,  0.0272,  0.1063, -0.0192],
          [-0.1297,  0.0393,  0.2007, -0.0076,  0.0877,  0.0571]]]],
       grad_fn=<CatBackward0>), (6, 3): tensor([[[[ 0.0596,  0.0594, -0.3174, -0.0639],
          [ 0.0059, -0.0067, -0.0793, -0.0610],
          [-0.0258, -0.0267,  

In [18]:
print(dataset_df.density_fitting[0]['df_coeff'].shape)
from equiv_dens.utils import orbitals
vectors = {}
basis = {}
for key in dataset_df.orbital_basis.keys():
    z = utils.symbols_to_numbers(key)
    basis[z[0]] = dataset_df.orbital_basis[key]
vectors['spherical_coeffs'] = torch.tensor(dataset_df.density_fitting[0]['df_coeff']).unsqueeze(0)
coeffs = orbitals.vector_to_coeffs_dict(vectors, basis, torch.tensor(dataset_df.atoms['atom_numbers']).unsqueeze(0), radial_coeffs=False)
print(coeffs)

(882,)
{'spherical_coeffs': [{(6, 0): tensor([[[[ 1.3504e-01,  3.3099e-01,  1.1488e+00,  1.7689e+00,  2.2057e+00,
            6.3295e-01, -1.8578e-01,  3.8899e-01,  1.6697e-01,  3.4432e-01,
            1.6181e-03]]]], dtype=torch.float64), (6, 1): tensor([[[[ 6.9307e-04,  1.5629e-03,  5.1718e-03, -2.4129e-03, -2.8663e-03,
           -1.9556e-02,  7.5086e-02, -3.7786e-03],
          [-9.7481e-05,  3.8117e-04, -2.1884e-04,  4.3205e-03, -8.1919e-03,
           -1.8460e-03,  8.4099e-02,  1.4121e-03],
          [-6.2382e-04, -2.9568e-04, -3.7928e-03,  4.3132e-03, -6.1231e-03,
            2.4876e-02, -3.5606e-02,  1.9652e-03]]]], dtype=torch.float64), (6, 2): tensor([[[[-1.2122e-03, -5.1023e-03, -2.7758e-03, -3.8059e-02, -8.0417e-03,
           -1.9614e-03],
          [ 7.3582e-04,  2.1288e-03, -1.3312e-03,  2.5978e-02, -3.3401e-03,
           -3.6294e-04],
          [ 6.1395e-04,  2.4767e-03,  3.4103e-03, -2.9044e-03, -5.2508e-02,
           -3.4582e-04],
          [ 1.6875e-03,  8.3646e-03